# Example 2 · Cat-Comm vs TP1 Across Transducer Noise Levels

The paper implements two remote-gate protocols — Cat-Comm and TP1 — and
shows results at a fixed operating point (κ_T = 0.5).  This example runs
both protocols across a range of transducer noise levels from low to high,
on the same physical qubits under the same conditions, and compares their
success probabilities.

The result shows which protocol performs better at each noise level and
where the two curves cross — giving a concrete answer to the practical
question of when to use each protocol in a real QDC.


## Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from QdcEm import Algorithms, RemoteGates, QPU
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from math import sqrt

from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import json, pathlib

# ── Backend ───────────────────────────────────────────────────────────────────
_creds  = json.loads(pathlib.Path("../ibm_credentials.json").read_text())
service = QiskitRuntimeService(channel=_creds["channel"],
                               instance=_creds["instance"],
                               token=_creds["token"])
backend = service.backend("ibm_torino")
_aer    = AerSimulator.from_backend(backend)

# ── QPU layout ────────────────────────────────────────────────────────────────
#   q[0]=CommA(3)  q[1]=ENA(14)  q[2]=Proc_A(0)
#   q[3]=CommB(4)  q[4]=ENB(15)  q[5]=Proc_B(7)
QPUA = QPU.Make(Comm=3,  EN=14, Processing_Qubits=[0])
QPUB = QPU.Make(Comm=4,  EN=15, Processing_Qubits=[7])
QPUs = [QPUA, QPUB]

shots       = 10_000
simulator   = False
fiber_steps = 3                          # fixed: 30 m of G-654-E fiber
kappa_F     = sqrt(0.01 * 0.0392)
kT_sweep    = np.linspace(0.05, 0.80, 16)

print(f"Sweeping {len(kT_sweep)} κ_T values, fiber_steps={fiber_steps} (30 m G-654-E)")
print(f"Total circuits: {len(kT_sweep) * 2}  (Cat-Comm + TP1 per point)")


## Run both protocols across the κ_T sweep

In [ ]:
my_backend = _aer if simulator else backend
sampler    = SamplerV2(mode=my_backend)

success_cat = []
success_tp1 = []

for kT in kT_sweep:
    for protocol, results_list in [('cat', success_cat), ('tp1', success_tp1)]:

        q  = QuantumRegister(6, name='q')
        c  = ClassicalRegister(2, name='c')
        qc = QuantumCircuit(q, c)

        qc.x(q[2])   # control qubit |1⟩ — more sensitive to noise

        if protocol == 'cat':
            RemoteGates.remote_cx(
                qc,
                control=q[2], target=q[5],
                CommA=q[0], ENA=q[1], CommB=q[3], ENB=q[4],
                creg=c, creg_index=0,
                kappa_Fiber=kappa_F, Steps=fiber_steps - 1,
                kappa_Transductor=kT,
            )
            qc.measure(q[2], c[0])
            qc.measure(q[5], c[1])
        else:
            RemoteGates.remote_cx_TP1(
                qc,
                control=q[2], target=q[5],
                CommA=q[0], ENA=q[1], CommB=q[3], ENB=q[4],
                creg=c, creg_index=0,
                kappa_Fiber=kappa_F, Steps=fiber_steps - 1,
                kappa_Transductor=kT,
            )
            qc.measure(q[3], c[0])   # CommB holds teleported control in TP1
            qc.measure(q[5], c[1])

        initial_layout = QPU.Get_Initial_Layout(QPUs, QRG=q)
        pm = generate_preset_pass_manager(optimization_level=3,
                                          target=my_backend.target,
                                          initial_layout=initial_layout)
        result = sampler.run([pm.run(qc)], shots=shots)
        print(f"  [κ_T={kT:.2f}  {protocol.upper():7s}]  job={result.job_id()}")

        data   = result.result()[0].data
        attr   = next(iter(vars(data)))
        counts = getattr(data, attr).get_counts()
        bc     = defaultdict(int)
        for bs, cnt in counts.items():
            bc[''.join(bs[p] for p in (1, 0))] += cnt

        results_list.append(bc.get('11', 0) / shots)

success_cat = np.array(success_cat)
success_tp1 = np.array(success_tp1)
print("\nDone.")


## Find the crossover point and plot

In [ ]:
# ── Crossover: where TP1 and Cat-Comm swap ranking ───────────────────────────
diff          = success_tp1 - success_cat
crossover_kT  = None
for k in range(len(diff) - 1):
    if diff[k] * diff[k+1] < 0:
        x0, x1 = kT_sweep[k], kT_sweep[k+1]
        crossover_kT = x0 - diff[k] * (x1 - x0) / (diff[k+1] - diff[k])
        break

# ── Plot ──────────────────────────────────────────────────────────────────────
plt.rcParams.update({'font.family': 'serif', 'font.size': 14})
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(kT_sweep, success_cat, 'o-',  color='royalblue', lw=2, label='Cat-Comm')
ax.plot(kT_sweep, success_tp1, 's--', color='tomato',    lw=2, label='TP1')

if crossover_kT is not None:
    ax.axvline(crossover_kT, color='grey', linestyle=':', lw=1.5)
    cross_prob = float(np.interp(crossover_kT, kT_sweep,
                                 (success_cat + success_tp1) / 2))
    ax.annotate(f'$\kappa_T^* \approx {crossover_kT:.2f}$',
                xy=(crossover_kT, cross_prob),
                xytext=(crossover_kT + 0.04, cross_prob + 0.05),
                arrowprops=dict(arrowstyle='->', color='grey'),
                fontsize=12, color='dimgrey')
    ax.axvspan(kT_sweep[0],  crossover_kT, alpha=0.06, color='royalblue')
    ax.axvspan(crossover_kT, kT_sweep[-1], alpha=0.06, color='tomato')

# Mark the paper's operating point
ax.axvline(0.5, color='black', linestyle='--', lw=1, alpha=0.4)
ax.text(0.52, 0.08, 'Paper
operating
point', fontsize=9,
        color='black', alpha=0.6)

ax.set_xlabel("Transducer noise κ_T")
ax.set_ylabel("Success probability")
ax.set_title(
    "Cat-Comm vs TP1  |  control $|1\rangle$  |  "
    f"fiber steps = {fiber_steps} (30 m, G-654-E)",
    fontsize=12
)
ax.set_ylim(0, 1)
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig("protocol_race.png", dpi=300, bbox_inches='tight')
plt.show()

if crossover_kT:
    print(f"Crossover at κ_T* ≈ {crossover_kT:.3f}")
    print(f"  κ_T < {crossover_kT:.2f} → Cat-Comm performs better")
    print(f"  κ_T > {crossover_kT:.2f} → TP1 performs better")
